# 1. Initial Model Load

In this notebook, we:
- Detect the appropriate device (MPS, CUDA, or CPU)
- Load the LLM model and tokenizer in float16 precision to minimize memory usage
- Download and prepare the WikiText-2 dataset for future finetuning tasks
- Save the initial model checkpoint for subsequent notebooks

**Note:** We are ensuring all parameters are fully specified (nothing is left as optional) to meet our project’s compression and resource goals.


In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

## Device Setup

We use MPS (for macOS systems) if available; otherwise, the notebook falls back to CUDA (if available) or CPU.


In [2]:
def get_device():
    # Prefer MPS if available (for Apple Silicon/Mac)
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Device Selected: MPS (Apple Silicon)")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("Device Selected: CUDA")
    else:
        device = torch.device("cpu")
        print("Device Selected: CPU")
    return device

device = get_device()

Device Selected: MPS (Apple Silicon)


## Configuration

Here we define the model and dataset identifiers. We have chosen `gpt2` as a placeholder model which is small enough to meet our memory constraints. You can swap this out with any Hugging Face model that respects your 25 GB footprint requirement.

We also select WikiText-2 (raw version) as our dataset. This dataset is lightweight and widely used for language model evaluation and finetuning.


In [3]:
# Configuration parameters
MODEL_NAME = "openai-community/gpt2"  # Replace with your target model identifier if needed.
DATASET_NAME = "wikitext"
DATASET_CONFIG_NAME = "wikitext-2-raw-v1"  # Raw version for direct text loading


## Model and Tokenizer Loading

The model is loaded in `float16` using the `torch_dtype=torch.float16` flag to reduce memory overhead. It is then moved to the detected device.


In [4]:
print("Loading model and tokenizer in float16 mode...")
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,  # Ensure lower precision
        low_cpu_mem_usage=True  # Helps reduce memory footprint during loading
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # Move model to the selected device
    model.to(device)
    print("Model and tokenizer loaded successfully.")
except Exception as e:
    print(f"Error during model/tokenizer loading: {e}")
    raise

Loading model and tokenizer in float16 mode...
Model and tokenizer loaded successfully.


## Dataset Loading

We load the WikiText-2 dataset using Hugging Face’s `datasets` library. This dataset is ideal for evaluation and finetuning while remaining lightweight.



In [5]:
print("Loading dataset...")
try:
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG_NAME)
    # Optionally, you can preview the dataset splits:
    print("Available splits:", list(dataset.keys()))
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Error during dataset loading: {e}")
    raise

Loading dataset...
Available splits: ['test', 'train', 'validation']
Dataset loaded successfully.


## Saving the Initial Checkpoint

To ensure reproducibility across our pipeline (quantization, fine-tuning, pruning, etc.), we save the loaded model and tokenizer. This checkpoint will be used in subsequent notebooks.


In [6]:
CHECKPOINT_DIR = "../checkpoints/initial_model"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Saving initial checkpoint...")
try:
    # The save_pretrained method ensures that both the model and tokenizer configurations are saved.
    model.save_pretrained(CHECKPOINT_DIR)
    tokenizer.save_pretrained(CHECKPOINT_DIR)
    print(f"Initial checkpoint saved successfully in '{CHECKPOINT_DIR}/'")
except Exception as e:
    print(f"Error saving checkpoint: {e}")
    raise

Saving initial checkpoint...
Initial checkpoint saved successfully in '../checkpoints/initial_model/'


In [7]:
# %% [code]
import os

DATA_DIR = "../data/wikitext-2"
os.makedirs(DATA_DIR, exist_ok=True)

# Save the dataset to disk in the specified directory
dataset.save_to_disk(DATA_DIR)
print(f"Dataset saved successfully to {DATA_DIR}.")


Saving the dataset (0/1 shards):   0%|          | 0/4358 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/36718 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset saved successfully to ../data/wikitext-2.


## Completion Message

In [8]:
print("1-initial_model_load notebook execution complete. The model and dataset are now ready for further processing in the pipeline.")

1-initial_model_load notebook execution complete. The model and dataset are now ready for further processing in the pipeline.
